In [1]:
# utils/extract.py
import requests
import time
import pandas as pd
from logger import get_logger
import great_expectations as ge

KeyboardInterrupt: 

In [ ]:
logger = get_logger("extract")

API_URL = "https://api.open-meteo.com/v1/forecast"

LOCATIONS = [
    {"city": "Lagos", "lat": 6.5244, "lon": 3.3792},
    {"city": "London", "lat": 51.5074, "lon": -0.1278},
    {"city": "New York", "lat": 40.7128, "lon": -74.0060},
]

MAX_RETRIES = 3
RETRY_DELAY = 5

In [ ]:
def extract_weather():
    all_data = []

    for loc in LOCATIONS:
        city = loc["city"]
        latitude = loc["lat"]
        longitude = loc["lon"]

        logger.info(f"Starting extraction for {city}")

        params = {
            "latitude": latitude,
            "longitude": longitude,
            "hourly": "temperature_2m,relativehumidity_2m,windspeed_10m,precipitation"
        }

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = requests.get(API_URL, params=params, timeout=10)
                response.raise_for_status()

                data = response.json()

                df = pd.DataFrame({
                    "city": city,
                    "timestamp": data["hourly"]["time"],
                    "temperature": data["hourly"]["temperature_2m"],
                    "humidity": data["hourly"]["relativehumidity_2m"],
                    "windspeed": data["hourly"]["windspeed_10m"],
                    "precipitation": data["hourly"]["precipitation"],
                })

                logger.info(f"{city}: Extracted {len(df)} records")
                all_data.append(df)
                break

            except Exception as e:
                logger.error(f"{city} attempt {attempt} failed: {e}")

                if attempt < MAX_RETRIES:
                    logger.info(f"{city}: Retrying in {RETRY_DELAY} seconds...")
                    time.sleep(RETRY_DELAY)
                else:
                    logger.error(f"{city}: All retries failed")
                    raise

    final_df = pd.concat(all_data, ignore_index=True)
    return final_df


In [ ]:
df = extract_weather()

2026-01-18 14:22:32,359 | INFO | extract | Starting extraction for Lagos
2026-01-18 14:22:34,271 | INFO | extract | Lagos: Extracted 168 records
2026-01-18 14:22:34,271 | INFO | extract | Starting extraction for London
2026-01-18 14:22:35,499 | INFO | extract | London: Extracted 168 records
2026-01-18 14:22:35,501 | INFO | extract | Starting extraction for New York
2026-01-18 14:22:36,524 | INFO | extract | New York: Extracted 168 records


In [ ]:
print(df)

         city         timestamp  temperature  humidity  windspeed  \
0       Lagos  2026-01-18T00:00         27.7        89        4.1   
1       Lagos  2026-01-18T01:00         27.5        89        4.3   
2       Lagos  2026-01-18T02:00         27.3        90        4.9   
3       Lagos  2026-01-18T03:00         27.2        92        4.9   
4       Lagos  2026-01-18T04:00         27.1        92        5.1   
..        ...               ...          ...       ...        ...   
499  New York  2026-01-24T19:00         -8.5        56        8.8   
500  New York  2026-01-24T20:00         -8.3        56        7.6   
501  New York  2026-01-24T21:00         -8.2        58        7.1   
502  New York  2026-01-24T22:00         -8.2        62        7.4   
503  New York  2026-01-24T23:00         -8.3        67        8.5   

     precipitation  
0              0.0  
1              0.0  
2              0.0  
3              0.0  
4              0.0  
..             ...  
499            0.0  
500

### TRANSFORM

In [ ]:
# utils/transform.py

logger = get_logger("transform")

In [ ]:
def transform_weather(df: pd.DataFrame) -> pd.DataFrame:
    logger.info("Starting transformation process")

    # Convert timestamp
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)

    # Drop rows with invalid timestamps
    before = len(df)
    df = df.dropna(subset=["timestamp"])
    logger.info(f"Dropped {before - len(df)} rows with invalid timestamps")

    # Filter unrealistic values
    df = df[
        (df["temperature"].between(-60, 60)) &
        (df["humidity"].between(0, 100)) &
        (df["windspeed"] >= 0) &
        (df["precipitation"] >= 0)
    ]

    logger.info(f"Records after cleaning: {len(df)}")

    # Enrichment
    df["is_raining"] = df["precipitation"] > 0

    df["comfort_level"] = df["temperature"].apply(
        lambda x: "Cold" if x < 15 else "Hot" if x > 30 else "Comfortable"
    )

    df["heat_index"] = df["temperature"] + (df["humidity"] * 0.05)

    logger.info("Transformation and enrichment completed")

    return df

In [ ]:
transform_weather(df)

2026-01-18 14:22:36,608 | INFO | transform | Starting transformation process
2026-01-18 14:22:36,616 | INFO | transform | Dropped 0 rows with invalid timestamps
2026-01-18 14:22:36,627 | INFO | transform | Records after cleaning: 504
2026-01-18 14:22:36,636 | INFO | transform | Transformation and enrichment completed


,city,timestamp,temperature,humidity,windspeed,precipitation,is_raining,comfort_level,heat_index
0,Lagos,2026-01-18 00:00:00+00:00,27.7,89,4.1,0.0,False,Comfortable,32.15
1,Lagos,2026-01-18 01:00:00+00:00,27.5,89,4.3,0.0,False,Comfortable,31.95
2,Lagos,2026-01-18 02:00:00+00:00,27.3,90,4.9,0.0,False,Comfortable,31.80
3,Lagos,2026-01-18 03:00:00+00:00,27.2,92,4.9,0.0,False,Comfortable,31.80
4,Lagos,2026-01-18 04:00:00+00:00,27.1,92,5.1,0.0,False,Comfortable,31.70
...,...,...,...,...,...,...,...,...,...
499,New York,2026-01-24 19:00:00+00:00,-8.5,56,8.8,0.0,False,Cold,-5.70
500,New York,2026-01-24 20:00:00+00:00,-8.3,56,7.6,0.0,False,Cold,-5.50
501,New York,2026-01-24 21:00:00+00:00,-8.2,58,7.1,0.0,False,Cold,-5.30
502,New York,2026-01-24 22:00:00+00:00,-8.2,62,7.4,0.0,False,Cold,-5.10


### Great Expectation for data quality

In [ ]:
logger = get_logger("data_quality")

In [ ]:
def validate_weather(df):
    logger.info("Starting data quality checks")

    ge_df = ge.from_pandas(df)

    ge_df.expect_column_values_to_not_be_null("timestamp")
    ge_df.expect_column_values_to_not_be_null("city")

    ge_df.expect_column_values_to_be_between("temperature", -60, 60)
    ge_df.expect_column_values_to_be_between("humidity", 0, 100)
    ge_df.expect_column_values_to_be_between("windspeed", 0, 300)
    ge_df.expect_column_values_to_be_between("precipitation", 0, 500)

    result = ge_df.validate()

    if not result["success"]:
        logger.error("Data quality checks failed")
        raise ValueError("Data quality validation failed")

    logger.info("Data quality checks passed successfully")

    return df